# Notebook 01 — Build Master Patient Index (MPI)

**Goal:** Extract the core patient identities from Synthea and enrich each patient with:
- Their primary active clinical condition (for MTSamples and MedMNIST matching)
- Their primary active medication (for RxHandBD matching)
- Their imaging modality (for MedMNIST folder matching)

**Output:** `data_preparation/linked/patients_master.csv`

## 1. Imports & Paths

In [1]:
import pandas as pd
import numpy as np
import random
import os

SYNTHEA_DIR = "../raw/synthea/"
OUTPUT_DIR  = "../linked/"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Paths OK")

Paths OK


## 2. Load Synthea Core Files

In [2]:
patients        = pd.read_csv(os.path.join(SYNTHEA_DIR, "patients.csv"))
conditions      = pd.read_csv(os.path.join(SYNTHEA_DIR, "conditions.csv"))
medications     = pd.read_csv(os.path.join(SYNTHEA_DIR, "medications.csv"))
imaging_studies = pd.read_csv(os.path.join(SYNTHEA_DIR, "imaging_studies.csv"))

print(f"Patients        : {len(patients)} rows")
print(f"Conditions      : {len(conditions)} rows")
print(f"Medications     : {len(medications)} rows")
print(f"Imaging Studies : {len(imaging_studies)} rows")

Patients        : 1163 rows
Conditions      : 38094 rows
Medications     : 56430 rows
Imaging Studies : 151637 rows


## 3. Build Base MPI from patients.csv

In [3]:
mpi = patients[[
    "Id", "BIRTHDATE", "DEATHDATE", "PREFIX", "FIRST", "LAST",
    "GENDER", "RACE", "ETHNICITY", "BIRTHPLACE",
    "CITY", "STATE", "HEALTHCARE_EXPENSES", "HEALTHCARE_COVERAGE"
]].copy()

# Rename Id to patient_id
mpi.rename(columns={"Id": "patient_id"}, inplace=True)

# Compute age from birthdate
mpi["BIRTHDATE"] = pd.to_datetime(mpi["BIRTHDATE"])
mpi["age"] = ((pd.Timestamp("today") - mpi["BIRTHDATE"]).dt.days / 365.25).astype(int)

# Flag deceased patients
mpi["is_deceased"] = mpi["DEATHDATE"].notna()

print(f"MPI base built: {len(mpi)} patients")
mpi.head(3)

MPI base built: 1163 patients


,patient_id,BIRTHDATE,DEATHDATE,PREFIX,FIRST,LAST,GENDER,RACE,ETHNICITY,BIRTHPLACE,CITY,STATE,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,age,is_deceased
0,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,2019-02-17,NaN,NaN,Damon455,Langosh790,M,white,nonhispanic,Middleborough Massachusetts US,Springfield,Massachusetts,9039.1645,7964.1255,7,False
1,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,2005-07-04,NaN,NaN,Thi53,Wunsch504,F,white,nonhispanic,Danvers Massachusetts US,Bellingham,Massachusetts,402723.4150,14064.1350,20,False
2,339144f8-50e1-633e-a013-f361391c4cff,1998-05-11,NaN,Mr.,Chi716,Greenfelder433,M,white,nonhispanic,Athens Athens Prefecture GR,Boston,Massachusetts,571935.8725,787.5375,27,False


## 4. Derive Primary Clinical Condition per Patient

In [4]:
# Filter out non-clinical social/employment findings
EXCLUDE_KEYWORDS = [
    "employment", "education", "labor force", "social", "housing",
    "criminal", "violence", "abuse", "alcohol", "risk activity",
    "pregnancy", "school", "certificate", "higher education",
    "primary school", "isolation", "contact", "stress",
    "transport", "access to", "labor"
]

def is_clinical(description):
    desc_lower = str(description).lower()
    return not any(kw in desc_lower for kw in EXCLUDE_KEYWORDS)

clinical_conditions = conditions[conditions["DESCRIPTION"].apply(is_clinical)].copy()

# Active clinical conditions (STOP is null = still ongoing)
active_clinical = clinical_conditions[clinical_conditions["STOP"].isna()].copy()
active_clinical["START"] = pd.to_datetime(active_clinical["START"])
active_clinical = active_clinical.sort_values("START", ascending=False)
primary_condition = active_clinical.groupby("PATIENT").first()[["DESCRIPTION"]].reset_index()
primary_condition.columns = ["patient_id", "primary_condition"]

# Fallback — most recent clinical condition regardless of active status
all_clinical = clinical_conditions.copy()
all_clinical["START"] = pd.to_datetime(all_clinical["START"])
all_clinical = all_clinical.sort_values("START", ascending=False)
fallback_condition = all_clinical.groupby("PATIENT").first()[["DESCRIPTION"]].reset_index()
fallback_condition.columns = ["patient_id", "fallback_condition"]

print(f"Patients with active clinical condition : {len(primary_condition)}")
print()
print("Top 10 primary conditions:")
print(primary_condition["primary_condition"].value_counts().head(10))

Patients with active clinical condition : 950

Top 10 primary conditions:
primary_condition
Body mass index 30+ - obesity (finding)    94
Chronic low back pain (finding)            77
Chronic sinusitis (disorder)               50
Anemia (disorder)                          44
Hypertension                               42
Prediabetes                                41
Chronic neck pain (finding)                38
Miscarriage in first trimester             32
Hyperlipidemia                             32
Stroke                                     26
Name: count, dtype: int64


## 5. Derive Primary Medication per Patient

In [5]:
# Active medications (STOP is null)
active_meds = medications[medications["STOP"].isna()].copy()
active_meds["START"] = pd.to_datetime(active_meds["START"])
active_meds = active_meds.sort_values("START", ascending=False)
primary_medication = active_meds.groupby("PATIENT").first()[["DESCRIPTION"]].reset_index()
primary_medication.columns = ["patient_id", "primary_medication"]

# Fallback — most recent medication regardless of active status
all_meds = medications.copy()
all_meds["START"] = pd.to_datetime(all_meds["START"])
all_meds = all_meds.sort_values("START", ascending=False)
fallback_medication = all_meds.groupby("PATIENT").first()[["DESCRIPTION"]].reset_index()
fallback_medication.columns = ["patient_id", "fallback_medication"]

print(f"Patients with active medication : {len(primary_medication)}")
print()
print("Top 10 primary medications:")
print(primary_medication["primary_medication"].value_counts().head(10))

Patients with active medication : 815

Top 10 primary medications:
primary_medication
Simvastatin 10 MG Oral Tablet                                         77
lisinopril 10 MG Oral Tablet                                          71
Hydrochlorothiazide 25 MG Oral Tablet                                 68
amLODIPine 2.5 MG Oral Tablet                                         57
NDA020800 0.3 ML Epinephrine 1 MG/ML Auto-Injector                    25
Acetaminophen 325 MG Oral Tablet [Tylenol]                            24
Ibuprofen 400 MG Oral Tablet [Ibu]                                    24
Acetaminophen 300 MG / Hydrocodone Bitartrate 5 MG Oral Tablet        21
24 HR Metformin hydrochloride 500 MG Extended Release Oral Tablet     19
NDA020503 200 ACTUAT Albuterol 0.09 MG/ACTUAT Metered Dose Inhaler    17
Name: count, dtype: int64


## 6. Derive Primary Imaging Modality per Patient

In [6]:
# Map Synthea modality/bodysite combinations to MedMNIST folder names
def map_to_medmnist(modality, bodysite):
    modality = str(modality).upper()
    bodysite  = str(bodysite).upper()

    if "COMPUTED TOMOGRAPHY" in modality:
        if "THORAC" in bodysite or "CHEST" in bodysite:
            return "ChestCT"
        elif "ABDOMEN" in bodysite or "ABDOM" in bodysite or "QUADRANT" in bodysite:
            return "AbdomenCT"
        else:
            return "HeadCT"
    elif "DIGITAL RADIOGRAPHY" in modality or "COMPUTED RADIOGRAPHY" in modality:
        if "HAND" in bodysite or "WRIST" in bodysite or "FINGER" in bodysite:
            return "Hand"
        else:
            return "CXR"
    elif "MR" in modality or "MAGNETIC" in modality:
        return "BreastMRI"
    else:
        return None  # Unknown — handled by gender-aware assignment below

imaging_studies["DATE"] = pd.to_datetime(imaging_studies["DATE"])
imaging_sorted = imaging_studies.sort_values("DATE", ascending=False)
primary_imaging = imaging_sorted.groupby("PATIENT").first()[[
    "MODALITY_DESCRIPTION", "BODYSITE_DESCRIPTION"
]].reset_index()
primary_imaging.columns = ["patient_id", "imaging_modality", "imaging_bodysite"]

primary_imaging["medmnist_folder"] = primary_imaging.apply(
    lambda row: map_to_medmnist(row["imaging_modality"], row["imaging_bodysite"]), axis=1
)

print(f"Patients with imaging record : {len(primary_imaging)}")
print()
print("MedMNIST folder distribution (real imaging only):")
print(primary_imaging["medmnist_folder"].value_counts())

Patients with imaging record : 238

MedMNIST folder distribution (real imaging only):
medmnist_folder
CXR        164
Hand        42
ChestCT      8
Name: count, dtype: int64


## 7. Merge Everything into Final MPI

In [7]:
mpi_full = mpi.copy()

# Merge conditions
mpi_full = mpi_full.merge(primary_condition,  on="patient_id", how="left")
mpi_full = mpi_full.merge(fallback_condition, on="patient_id", how="left")
mpi_full["primary_condition"] = mpi_full["primary_condition"].fillna(mpi_full["fallback_condition"])
mpi_full.drop(columns=["fallback_condition"], inplace=True)

# Merge medications
mpi_full = mpi_full.merge(primary_medication,  on="patient_id", how="left")
mpi_full = mpi_full.merge(fallback_medication, on="patient_id", how="left")
mpi_full["primary_medication"] = mpi_full["primary_medication"].fillna(mpi_full["fallback_medication"])
mpi_full.drop(columns=["fallback_medication"], inplace=True)

# Merge imaging
mpi_full = mpi_full.merge(
    primary_imaging[["patient_id", "imaging_modality", "imaging_bodysite", "medmnist_folder"]],
    on="patient_id", how="left"
)

print(f"Merged successfully — Shape: {mpi_full.shape}")

Merged successfully — Shape: (1163, 21)


## 8. Gender-Aware MedMNIST Folder Assignment

Patients without a real imaging record get a randomly assigned MedMNIST folder.
BreastMRI is only assigned to female patients for clinical coherence.

In [8]:
def assign_medmnist_folder(row):
    # Keep real imaging assignment if exists
    if pd.notna(row["medmnist_folder"]):
        return row["medmnist_folder"]
    # Gender-aware random assignment
    if row["GENDER"] == "F":
        folders = ["CXR", "ChestCT", "AbdomenCT", "HeadCT", "BreastMRI", "Hand"]
    else:
        folders = ["CXR", "ChestCT", "AbdomenCT", "HeadCT", "Hand"]
    return random.choice(folders)

mpi_full["medmnist_folder"] = mpi_full.apply(assign_medmnist_folder, axis=1)

print("MedMNIST folder distribution by gender:")
print(mpi_full.groupby(["GENDER", "medmnist_folder"]).size().unstack(fill_value=0))

MedMNIST folder distribution by gender:
medmnist_folder  AbdomenCT  BreastMRI  CXR  ChestCT  Hand  HeadCT
GENDER                                                           
F                       78         85  172       90    96      95
M                       97          0  171       83   120      76


## 9. Final Quality Check

In [9]:
print("=== Final MPI Quality Check ===")
print(f"Total patients            : {len(mpi_full)}")
print(f"With primary condition    : {mpi_full['primary_condition'].notna().sum()}")
print(f"With primary medication   : {mpi_full['primary_medication'].notna().sum()}")
print(f"With real imaging record  : {mpi_full['imaging_modality'].notna().sum()}")
print()
print("Gender distribution:")
print(mpi_full["GENDER"].value_counts())
print()
print("MedMNIST folder distribution (all patients):")
print(mpi_full["medmnist_folder"].value_counts())
print()
print("BreastMRI assigned to male patients (must be 0):")
print(len(mpi_full[(mpi_full["GENDER"] == "M") & (mpi_full["medmnist_folder"] == "BreastMRI")]))
print()
print("Top 10 primary conditions:")
print(mpi_full["primary_condition"].value_counts().head(10))
print()
print("Top 10 primary medications:")
print(mpi_full["primary_medication"].value_counts().head(10))

=== Final MPI Quality Check ===
Total patients            : 1163
With primary condition    : 1145
With primary medication   : 1117
With real imaging record  : 238

Gender distribution:
GENDER
F    616
M    547
Name: count, dtype: int64

MedMNIST folder distribution (all patients):
medmnist_folder
CXR          343
Hand         216
AbdomenCT    175
ChestCT      173
HeadCT       171
BreastMRI     85
Name: count, dtype: int64

BreastMRI assigned to male patients (must be 0):
0

Top 10 primary conditions:
primary_condition
Body mass index 30+ - obesity (finding)    94
Chronic low back pain (finding)            77
Chronic sinusitis (disorder)               50
Viral sinusitis (disorder)                 48
Anemia (disorder)                          45
Hypertension                               42
Prediabetes                                41
Chronic neck pain (finding)                38
Acute viral pharyngitis (disorder)         37
Otitis media                               34
Name: count, dty

## 10. Save Output

In [10]:
output_path = os.path.join(OUTPUT_DIR, "patients_master.csv")
mpi_full.to_csv(output_path, index=False)

print(f"Saved → {output_path}")
print(f"Shape  : {mpi_full.shape}")
print()
print("Final columns:")
print(mpi_full.columns.tolist())

Saved → ../linked/patients_master.csv
Shape  : (1163, 21)

Final columns:
['patient_id', 'BIRTHDATE', 'DEATHDATE', 'PREFIX', 'FIRST', 'LAST', 'GENDER', 'RACE', 'ETHNICITY', 'BIRTHPLACE', 'CITY', 'STATE', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'age', 'is_deceased', 'primary_condition', 'primary_medication', 'imaging_modality', 'imaging_bodysite', 'medmnist_folder']
